# INSTALLATIONS

In [ ]:
!pip install datasets
!pip install transformers datasets accelerate torch
!pip install bitsandbytes

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import sys
import re

In [ ]:
import sys
import os
import json
import ast

In [ ]:
import importlib
import inspect
import builtins

In [ ]:
import traceback
import signal
import threading
import copy
from pathlib import Path
from typing import Dict, Any, List, Optional, Tuple

In [ ]:
import types

In [ ]:

import re
import glob
from difflib import get_close_matches

In [ ]:
from datasets import load_dataset

# DATA LOADING

- I thought of only extracting the task id's of the column which have failed and giving it.

### Humaneval

In [ ]:
from datasets import load_dataset
ds = load_dataset("openai/openai_humaneval")

README.md: 0.00B [00:00, ?B/s]

openai_humaneval/test-00000-of-00001.par(…):   0%|          | 0.00/83.9k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/164 [00:00<?, ? examples/s]

In [ ]:
df_humaneval = pd.DataFrame(ds['test'])

In [ ]:
print(df_humaneval.columns)

Index(['task_id', 'prompt', 'canonical_solution', 'test', 'entry_point'], dtype='object')


### MBPP



In [ ]:
dataset = load_dataset("google-research-datasets/mbpp")
sanitized_dataset = load_dataset("google-research-datasets/mbpp", "sanitized")

README.md: 0.00B [00:00, ?B/s]

full/train-00000-of-00001.parquet:   0%|          | 0.00/87.2k [00:00<?, ?B/s]

full/test-00000-of-00001.parquet:   0%|          | 0.00/116k [00:00<?, ?B/s]

full/validation-00000-of-00001.parquet:   0%|          | 0.00/25.1k [00:00<?, ?B/s]

full/prompt-00000-of-00001.parquet:   0%|          | 0.00/7.88k [00:00<?, ?B/s]

...

sanitized/prompt-00000-of-00001.parquet:   0%|          | 0.00/6.72k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/120 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/257 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/43 [00:00<?, ? examples/s]

Generating prompt split:   0%|          | 0/7 [00:00<?, ? examples/s]

In [ ]:
mbpp_df = sanitized_dataset['train'].to_pandas()

In [ ]:
def extract_signature(code: str):
    for line in code.splitlines():
        line = line.strip()
        if line.startswith("def "):
            return line
    return None
mbpp_df['function_signature'] = mbpp_df['code'].apply(extract_signature)

In [ ]:
print(mbpp_df.columns)

Index(['source_file', 'task_id', 'prompt', 'code', 'test_imports', 'test_list',
       'function_signature'],
      dtype='object')


### DS1000

In [ ]:
ds1k = load_dataset("xlangai/DS-1000")
df_ds1k = ds1k['test'].to_pandas()
df_ds1k['prompt_2'] = df_ds1k['prompt'].str.split('A:', n=1).str[0].str.strip()

README.md:   0%|          | 0.00/554 [00:00<?, ?B/s]

test.jsonl: 0.00B [00:00, ?B/s]

Generating test split:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [ ]:
df_ds1k["task_id"] = [f"DS{str(i).zfill(3)}" for i in range(len(df_ds1k))]

In [ ]:
cols = ["task_id"] + [col for col in df_ds1k.columns if col != "task_id"]
df_ds1k = df_ds1k[cols]

In [ ]:
print(df_ds1k.columns)

Index(['task_id', 'prompt', 'reference_code', 'metadata', 'code_context',
       'prompt_2'],
      dtype='object')


In [ ]:
df_ds1k.to_csv("ds1k_NEW.csv", index=False)

In [ ]:
print('DS1000: ',df_ds1k.columns)
print('MBPP: ',mbpp_df.columns)
print('HumanEval: ',df_humaneval.columns)

DS1000:  Index(['task_id', 'prompt', 'reference_code', 'metadata', 'code_context',
       'prompt_2'],
      dtype='object')
MBPP:  Index(['source_file', 'task_id', 'prompt', 'code', 'test_imports', 'test_list',
       'function_signature'],
      dtype='object')
HumanEval:  Index(['task_id', 'prompt', 'canonical_solution', 'test', 'entry_point'], dtype='object')


### ERROR DATASET

- so here only erraneous code will be generated.

In [ ]:
task_id_ds1000 = ['DS0001', 'DS0002', 'DS0003', 'DS0004', 'DS0005', 'DS0006', 'DS0007', 'DS0008', 'DS0009', 'DS0010', 'DS0011', 'DS0012', 'DS0013', 'DS0014', 'DS0015', 'DS0016', 'DS0017', 'DS0018', 'DS0019', 'DS0020', 'DS0021', 'DS0022', 'DS0023', 'DS0024', 'DS0025', 'DS0026', 'DS0027', 'DS0028', 'DS0029', 'DS0032', 'DS0033', 'DS0034', 'DS0035', 'DS0038', 'DS0039', 'DS0040', 'DS0041', 'DS0042', 'DS0043', 'DS0044', 'DS0045', 'DS0046', 'DS0047', 'DS0048', 'DS0049', 'DS0053', 'DS0054', 'DS0055', 'DS0056', 'DS0057', 'DS0058', 'DS0059', 'DS0060', 'DS0061', 'DS0062', 'DS0063', 'DS0064', 'DS0065', 'DS0066', 'DS0067', 'DS0068', 'DS0069', 'DS0070', 'DS0071', 'DS0072', 'DS0073', 'DS0074', 'DS0075', 'DS0079', 'DS0080', 'DS0081', 'DS0082', 'DS0083', 'DS0084', 'DS0085', 'DS0086', 'DS0087', 'DS0088', 'DS0090', 'DS0091', 'DS0092', 'DS0093', 'DS0094', 'DS0095', 'DS0096', 'DS0097', 'DS0098', 'DS0099', 'DS0100', 'DS0101', 'DS0102', 'DS0103', 'DS0104', 'DS0105', 'DS0106', 'DS0107', 'DS0108', 'DS0109', 'DS0110', 'DS0111', 'DS0112', 'DS0113', 'DS0114', 'DS0115', 'DS0116', 'DS0117', 'DS0118', 'DS0119', 'DS0120', 'DS0121', 'DS0122', 'DS0123', 'DS0124', 'DS0125', 'DS0126', 'DS0127', 'DS0128', 'DS0129', 'DS0130', 'DS0131', 'DS0132', 'DS0133', 'DS0134', 'DS0136', 'DS0137', 'DS0140', 'DS0141', 'DS0142', 'DS0144', 'DS0148', 'DS0149', 'DS0150', 'DS0151', 'DS0152', 'DS0153', 'DS0154', 'DS0156', 'DS0157', 'DS0158', 'DS0159', 'DS0160', 'DS0161', 'DS0164', 'DS0165', 'DS0168', 'DS0169', 'DS0170', 'DS0171', 'DS0172', 'DS0173', 'DS0174', 'DS0178', 'DS0179', 'DS0181', 'DS0182', 'DS0183', 'DS0184', 'DS0185', 'DS0186', 'DS0187', 'DS0188', 'DS0189', 'DS0193', 'DS0195', 'DS0196', 'DS0197', 'DS0198', 'DS0199', 'DS0200', 'DS0201', 'DS0202', 'DS0203', 'DS0204', 'DS0205', 'DS0207', 'DS0208', 'DS0209', 'DS0210', 'DS0211', 'DS0212', 'DS0213', 'DS0214', 'DS0216', 'DS0219', 'DS0220', 'DS0221', 'DS0222', 'DS0223', 'DS0224', 'DS0225', 'DS0226', 'DS0227', 'DS0228', 'DS0229', 'DS0230', 'DS0231', 'DS0232', 'DS0233', 'DS0234', 'DS0235', 'DS0236', 'DS0237', 'DS0238', 'DS0239', 'DS0240', 'DS0241', 'DS0242', 'DS0243', 'DS0244', 'DS0245', 'DS0246', 'DS0247', 'DS0248', 'DS0249', 'DS0250', 'DS0251', 'DS0252', 'DS0253', 'DS0254', 'DS0255', 'DS0256', 'DS0257', 'DS0258', 'DS0259', 'DS0260', 'DS0262', 'DS0263', 'DS0264', 'DS0265', 'DS0266', 'DS0267', 'DS0268', 'DS0269', 'DS0270', 'DS0271', 'DS0272', 'DS0273', 'DS0274', 'DS0275', 'DS0278', 'DS0279', 'DS0280', 'DS0281', 'DS0282', 'DS0283', 'DS0284', 'DS0285', 'DS0286', 'DS0287', 'DS0288', 'DS0292', 'DS0293', 'DS0294', 'DS0296', 'DS0298', 'DS0299', 'DS0303', 'DS0304', 'DS0305', 'DS0306', 'DS0307', 'DS0308', 'DS0311', 'DS0312', 'DS0313', 'DS0315', 'DS0316', 'DS0318', 'DS0319', 'DS0322', 'DS0325', 'DS0328', 'DS0329', 'DS0330', 'DS0332', 'DS0333', 'DS0336', 'DS0337', 'DS0338', 'DS0339', 'DS0342', 'DS0345', 'DS0346', 'DS0347', 'DS0348', 'DS0349', 'DS0350', 'DS0351', 'DS0352', 'DS0353', 'DS0354', 'DS0355', 'DS0356', 'DS0357', 'DS0359', 'DS0360', 'DS0361', 'DS0362', 'DS0363', 'DS0364', 'DS0365', 'DS0366', 'DS0369', 'DS0370', 'DS0371', 'DS0372', 'DS0373', 'DS0374', 'DS0375', 'DS0376', 'DS0377', 'DS0378', 'DS0379', 'DS0383', 'DS0384', 'DS0385', 'DS0386', 'DS0387', 'DS0388', 'DS0389', 'DS0390', 'DS0391', 'DS0392', 'DS0394', 'DS0395', 'DS0396', 'DS0397', 'DS0398', 'DS0399', 'DS0402', 'DS0404', 'DS0406', 'DS0407', 'DS0408', 'DS0409', 'DS0410', 'DS0411', 'DS0414', 'DS0415', 'DS0416', 'DS0417', 'DS0418', 'DS0419', 'DS0420', 'DS0421', 'DS0422', 'DS0423', 'DS0424', 'DS0425', 'DS0426', 'DS0427', 'DS0430', 'DS0432', 'DS0433', 'DS0434', 'DS0435', 'DS0436', 'DS0437', 'DS0439', 'DS0440', 'DS0441', 'DS0442', 'DS0443', 'DS0444', 'DS0445', 'DS0446', 'DS0447', 'DS0450', 'DS0451', 'DS0452', 'DS0455', 'DS0456', 'DS0458', 'DS0459', 'DS0460', 'DS0461', 'DS0462', 'DS0463', 'DS0464', 'DS0465', 'DS0466', 'DS0467', 'DS0468', 'DS0469', 'DS0470', 'DS0474', 'DS0475', 'DS0476', 'DS0477', 'DS0478', 'DS0480', 'DS0482', 'DS0483', 'DS0484', 'DS0485', 'DS0486', 'DS0488', 'DS0489', 'DS0490', 'DS0491', 'DS0492', 'DS0494', 'DS0498', 'DS0499', 'DS0500', 'DS0501', 'DS0502', 'DS0503', 'DS0504', 'DS0505', 'DS0506', 'DS0507', 'DS0509', 'DS0510', 'DS0511', 'DS0512', 'DS0513', 'DS0514', 'DS0515', 'DS0516', 'DS0517', 'DS0520', 'DS0521', 'DS0523', 'DS0524', 'DS0526', 'DS0528', 'DS0529', 'DS0530', 'DS0533', 'DS0538', 'DS0540', 'DS0544', 'DS0548', 'DS0549', 'DS0550', 'DS0555', 'DS0556', 'DS0560', 'DS0563', 'DS0564', 'DS0565', 'DS0566', 'DS0567', 'DS0568', 'DS0569', 'DS0570', 'DS0571', 'DS0573', 'DS0574', 'DS0575', 'DS0576', 'DS0577', 'DS0578', 'DS0580', 'DS0582', 'DS0585', 'DS0586', 'DS0588', 'DS0590', 'DS0592', 'DS0593', 'DS0594', 'DS0595', 'DS0596', 'DS0597', 'DS0598', 'DS0599', 'DS0600', 'DS0601', 'DS0603', 'DS0604', 'DS0605', 'DS0606', 'DS0608', 'DS0609', 'DS0610', 'DS0612', 'DS0616', 'DS0617', 'DS0619', 'DS0620', 'DS0621', 'DS0622', 'DS0624', 'DS0625', 'DS0626', 'DS0629', 'DS0630', 'DS0633', 'DS0635', 'DS0637', 'DS0638', 'DS0640', 'DS0642', 'DS0644', 'DS0645', 'DS0646', 'DS0647', 'DS0649', 'DS0650', 'DS0651', 'DS0652', 'DS0654', 'DS0655', 'DS0656', 'DS0661', 'DS0662', 'DS0663', 'DS0665', 'DS0669', 'DS0670', 'DS0671', 'DS0672', 'DS0673', 'DS0674', 'DS0675', 'DS0676', 'DS0677', 'DS0678', 'DS0679', 'DS0680', 'DS0681', 'DS0682', 'DS0683', 'DS0684', 'DS0685', 'DS0686', 'DS0687', 'DS0688', 'DS0689', 'DS0690', 'DS0691', 'DS0692', 'DS0693', 'DS0694', 'DS0695', 'DS0696', 'DS0697', 'DS0698', 'DS0699', 'DS0700', 'DS0701', 'DS0702', 'DS0703', 'DS0704', 'DS0705', 'DS0706', 'DS0707', 'DS0708', 'DS0709', 'DS0712', 'DS0713', 'DS0714', 'DS0716', 'DS0717', 'DS0718', 'DS0719', 'DS0720', 'DS0721', 'DS0723', 'DS0724', 'DS0725', 'DS0726', 'DS0727', 'DS0728', 'DS0729', 'DS0730', 'DS0731', 'DS0732', 'DS0733', 'DS0734', 'DS0735', 'DS0736', 'DS0737', 'DS0738', 'DS0739', 'DS0740', 'DS0741', 'DS0742', 'DS0743', 'DS0744', 'DS0745', 'DS0746', 'DS0747', 'DS0748', 'DS0749', 'DS0750', 'DS0751', 'DS0752', 'DS0753', 'DS0754', 'DS0755', 'DS0759', 'DS0760', 'DS0761', 'DS0762', 'DS0763', 'DS0764', 'DS0765', 'DS0766', 'DS0767', 'DS0768', 'DS0770', 'DS0771', 'DS0772', 'DS0773', 'DS0774', 'DS0775', 'DS0776', 'DS0777', 'DS0778', 'DS0779', 'DS0780', 'DS0781', 'DS0782', 'DS0783', 'DS0784', 'DS0785', 'DS0786', 'DS0787', 'DS0789', 'DS0790', 'DS0792', 'DS0793', 'DS0794', 'DS0795', 'DS0796', 'DS0797', 'DS0798', 'DS0799', 'DS0800', 'DS0801', 'DS0802', 'DS0803', 'DS0805', 'DS0806', 'DS0807', 'DS0808', 'DS0809', 'DS0810', 'DS0811', 'DS0812', 'DS0813', 'DS0814', 'DS0815', 'DS0816', 'DS0817', 'DS0818', 'DS0819', 'DS0820', 'DS0821', 'DS0822', 'DS0823', 'DS0824', 'DS0825', 'DS0826', 'DS0827', 'DS0828', 'DS0829', 'DS0830', 'DS0831', 'DS0832', 'DS0833', 'DS0834', 'DS0835', 'DS0836', 'DS0837', 'DS0838', 'DS0839', 'DS0840', 'DS0842', 'DS0843', 'DS0844', 'DS0845', 'DS0846', 'DS0847', 'DS0848', 'DS0849', 'DS0850', 'DS0851', 'DS0852', 'DS0853', 'DS0855', 'DS0856', 'DS0857', 'DS0858', 'DS0859', 'DS0860', 'DS0861', 'DS0862', 'DS0863', 'DS0864', 'DS0865', 'DS0866', 'DS0867', 'DS0868', 'DS0869', 'DS0870', 'DS0871', 'DS0872', 'DS0873', 'DS0874', 'DS0875', 'DS0876', 'DS0877', 'DS0878', 'DS0879', 'DS0880', 'DS0881', 'DS0882', 'DS0883', 'DS0884', 'DS0885', 'DS0886', 'DS0887', 'DS0889', 'DS0892', 'DS0893', 'DS0894', 'DS0895', 'DS0896', 'DS0897', 'DS0898', 'DS0899', 'DS0900', 'DS0901', 'DS0902', 'DS0903', 'DS0904', 'DS0905', 'DS0906', 'DS0907', 'DS0908', 'DS0909', 'DS0910', 'DS0911', 'DS0914', 'DS0915', 'DS0916', 'DS0917', 'DS0918', 'DS0919', 'DS0920', 'DS0921', 'DS0922', 'DS0923', 'DS0924', 'DS0925', 'DS0926', 'DS0927', 'DS0928', 'DS0929', 'DS0931', 'DS0932', 'DS0933', 'DS0934', 'DS0935', 'DS0936', 'DS0937', 'DS0938', 'DS0939', 'DS0940', 'DS0941', 'DS0942', 'DS0943', 'DS0944', 'DS0945', 'DS0946', 'DS0947', 'DS0949', 'DS0950', 'DS0951', 'DS0952', 'DS0953', 'DS0954', 'DS0955', 'DS0956', 'DS0957', 'DS0958', 'DS0959', 'DS0960', 'DS0961', 'DS0962', 'DS0963', 'DS0964', 'DS0965', 'DS0966', 'DS0967', 'DS0968', 'DS0969', 'DS0970', 'DS0972', 'DS0974', 'DS0975', 'DS0976', 'DS0977', 'DS0978', 'DS0979', 'DS0980', 'DS0981', 'DS0982', 'DS0983', 'DS0984', 'DS0985', 'DS0986', 'DS0987', 'DS0988', 'DS0989', 'DS0990', 'DS0991', 'DS0992', 'DS0993', 'DS0994', 'DS0995', 'DS0996', 'DS0997', 'DS0998', 'DS0999']
task_id_humaneval = ['HumanEval/10', 'HumanEval/26', 'HumanEval/38', 'HumanEval/102', 'HumanEval/108', 'HumanEval/109', 'HumanEval/115', 'HumanEval/119', 'HumanEval/120', 'HumanEval/126', 'HumanEval/127', 'HumanEval/129', 'HumanEval/130', 'HumanEval/132', 'HumanEval/134', 'HumanEval/135', 'HumanEval/137', 'HumanEval/141', 'HumanEval/142', 'HumanEval/145', 'HumanEval/155', 'HumanEval/160', 'HumanEval/163', 'HumanEval/50', 'HumanEval/65', 'HumanEval/77', 'HumanEval/83', 'HumanEval/91', 'HumanEval/93', 'HumanEval/95', 'HumanEval/99']
task_id_mbpp = [603, 604, 607, 608, 610, 612, 615, 617, 619, 620, 622, 624, 626, 628, 629, 630, 631, 632, 638, 639, 640, 641, 643, 720, 721, 722, 725, 730, 732, 734, 788, 791, 796, 797, 800, 801, 802, 806, 808, 809, 735, 738, 739, 742, 747, 748, 751, 752, 753, 755, 756, 757, 758, 759, 763, 764, 765, 767, 769, 771, 772, 773, 776, 777, 778, 779, 780, 783, 784, 785, 11, 16, 18, 56, 59, 63, 64, 67, 70, 71, 74, 83, 84, 86, 87, 91, 92, 94, 102, 229, 230, 233, 235, 237, 244, 249, 251, 252, 253, 255, 259, 260, 262, 265, 268, 269, 272, 277, 279, 283, 284, 418, 419, 421, 424, 425, 427, 430, 431, 434, 437, 438, 440, 442, 443, 445, 446, 448, 450, 451, 452, 456, 457, 459, 461, 462, 464, 465, 468, 470, 473, 474, 475, 477, 478, 479, 103, 104, 106, 111, 117, 118, 119, 120, 124, 125, 128, 130, 132, 138, 143, 160, 162, 164, 226]


In [ ]:
df_ds1k_f = df_ds1k[df_ds1k["task_id"].isin(task_id_ds1000)]
df_mbpp_f = mbpp_df[mbpp_df["task_id"].isin(task_id_mbpp)]
df_humaneval_f = df_humaneval[df_humaneval["task_id"].isin(task_id_humaneval)]

In [ ]:
# Load ERROR_TYPE_IDS from error_type_ids.py (for error-type-wise runs)
import os
_error_type_ids_path = "AST+DYNMAIC+LIB_API/error_type_ids.py"
if os.path.exists(_error_type_ids_path):
    with open(_error_type_ids_path, encoding="utf-8") as f:
        exec(f.read())
else:
    ERROR_TYPE_IDS = {}

def get_dataset_config_for_error_type(error_type_filter):
    """Return DATASET_CONFIG-like dict filtered by error type, or None if not applicable."""
    if error_type_filter is None or error_type_filter not in ERROR_TYPE_IDS:
        return None
    ids = ERROR_TYPE_IDS[error_type_filter]
    config = {}
    if ids.get("humaneval"):
        df_h = df_humaneval[df_humaneval["task_id"].isin(ids["humaneval"])]
        if len(df_h) > 0:
            config["humaneval"] = {
                "df": df_h,
                "prompt_builder": lambda row: construct_prompt_humaneval(row["prompt"]),
                "code_extractor": extract_python_code_humaneval,
            }
    if ids.get("mbpp"):
        df_m = mbpp_df[mbpp_df["task_id"].isin(ids["mbpp"])]
        if len(df_m) > 0:
            config["mbpp"] = {
                "df": df_m,
                "prompt_builder": lambda row: construct_prompt_mbpp(row["prompt"], row["function_signature"]),
                "code_extractor": extract_python_code_humaneval,
            }
    if ids.get("ds1000"):
        df_d = df_ds1k[df_ds1k["task_id"].isin(ids["ds1000"])]
        if len(df_d) > 0:
            config["ds1000"] = {
                "df": df_d,
                "prompt_builder": lambda row: contruct_prompt_ds1k_v4(
                    row["prompt"],
                    extract_only_exec_context_wi(row["code_context"])
                ),
                "code_extractor": extract_python_code_humaneval,
            }
    return config if config else None

# AST

- INPUT :
```
AST_ANALYSIS_INPUT_SCHEMA = {
    "code": str  # Raw Python source code as a single string
}
```
- OUTPUT : ERROR INFORMATION
```
AST_ANALYSIS_OUTPUT_SCHEMA = {
    "ast_parsed": bool,
    "ast_errors": list[{
        "type": str,
        "start_line": int,
        "end_line": int,
        "col_offset": int | None,
        "message": str
    }]
}
```

- main function to be called `analyze_ast_for_patch`

In [ ]:
class StructuralViolationVisitor(ast.NodeVisitor):
    def __init__(self):
        self.errors = []
        self.in_function = 0
        self.in_loop = 0

    def _record(self, error_type: str, node: ast.AST):
        start = getattr(node, "lineno", None)
        end = getattr(node, "end_lineno", start)
        col = getattr(node, "col_offset", None)

        if start:
            self.errors.append({
                "type": error_type,
                "start_line": start,
                "end_line": end if end else start,
                "col_offset": col,
                "message": f"{error_type} detected"
            })

    def visit_FunctionDef(self, node):
        self.in_function += 1
        self.generic_visit(node)
        self.in_function -= 1

    def visit_AsyncFunctionDef(self, node):
        self.in_function += 1
        self.generic_visit(node)
        self.in_function -= 1

    def visit_For(self, node):
        self.in_loop += 1
        self.generic_visit(node)
        self.in_loop -= 1

    def visit_While(self, node):
        self.in_loop += 1
        self.generic_visit(node)
        self.in_loop -= 1

    def visit_Return(self, node):
        if self.in_function == 0:
            self._record("return_outside_function", node)
        self.generic_visit(node)

    def visit_Break(self, node):
        if self.in_loop == 0:
            self._record("break_outside_loop", node)

    def visit_Continue(self, node):
        if self.in_loop == 0:
            self._record("continue_outside_loop", node)


def analyze_ast_for_patch(code: str) -> Dict[str, Any]:
    """
    Clean AST analyzer for patch generation.
    """

    result = {
        "ast_parsed": False,
        "ast_errors": []
    }

    try:
        tree = ast.parse(code)
        result["ast_parsed"] = True

        visitor = StructuralViolationVisitor()
        visitor.visit(tree)

        result["ast_errors"].extend(visitor.errors)

    except IndentationError as e:
        result["ast_errors"].append({
            "type": "IndentationError",
            "start_line": e.lineno,
            "end_line": e.lineno,
            "col_offset": e.offset,
            "message": e.msg
        })

    except SyntaxError as e:
        result["ast_errors"].append({
            "type": "SyntaxError",
            "start_line": e.lineno,
            "end_line": e.lineno,
            "col_offset": e.offset,
            "message": e.msg
        })

    return result

# LIB_API


- Input : code string
```
LIB_API_ANALYSIS_INPUT_SCHEMA = {
    "code": str  
}
```
- Output : error information
```
LIB_API_ANALYSIS_OUTPUT_SCHEMA = {
    "libapi_analyzed": bool,

    "name_error": int,          # Invalid imported symbol name
    "attribute_error": int,     # Invalid attribute access
    "module_not_found": int,    # Always 0 (ignored intentionally)

    "total_libapi_errors": int,

    "libapi_details": list[{
        "type": str,            # "name_error" | "attribute_error"
        "name": str | None,     # Used when type == "name_error"
        "object": str | None,   # Base object/module
        "attribute": str | None,# Missing attribute
        "line": int             # Line number of issue
    }]
}

```
- Main function used to call is `analyze_library_api`

In [ ]:
BUILTINS = set(dir(builtins))


# =========================
# SAFE MODULE LOADER
# =========================

def safe_import_module(module_name):
    """
    # FIX: Ignore environment-dependent missing modules
    Only return module if actually importable.
    """
    try:
        return importlib.import_module(module_name)
    except ModuleNotFoundError:
        return None
    except Exception:
        # FIX: Ignore runtime import side-effects
        return None


# =========================
# VISITOR
# =========================

class LibraryAPIVistor(ast.NodeVisitor):
    def __init__(self):
        self.imports = {}
        self.errors = []

    # =========================
    # IMPORT HANDLING
    # =========================

    def visit_Import(self, node):
        for alias in node.names:
            module = safe_import_module(alias.name)

            if module is None:
                # FIX: DO NOT classify missing install as hallucination
                continue

            # FIX: Preserve full module name (no .split("."))
            name = alias.asname or alias.name
            self.imports[name] = module

    def visit_ImportFrom(self, node):
        if node.module is None:
            return

        module = safe_import_module(node.module)

        if module is None:
            # FIX: Ignore missing install
            return

        for alias in node.names:

            # FIX: Proper handling of import *
            if alias.name == "*":
                for attr in dir(module):
                    try:
                        self.imports[attr] = getattr(module, attr)
                    except Exception:
                        pass
                continue

            name = alias.asname or alias.name

            try:
                if hasattr(module, alias.name):
                    self.imports[name] = getattr(module, alias.name)
                else:
                    self.errors.append({
                        "type": "name_error",
                        "name": alias.name,
                        "line": node.lineno
                    })
            except Exception:
                # FIX: Avoid reflection crash
                pass

    # =========================
    # ATTRIBUTE RESOLUTION
    # =========================

    def resolve_attribute_chain(self, node):
        """
        # FIX: Proper chained attribute resolution
        Example:
        scipy.integrate.quad
        """
        parts = []

        while isinstance(node, ast.Attribute):
            parts.append(node.attr)
            node = node.value

        if isinstance(node, ast.Name):
            parts.append(node.id)
        else:
            return None

        return list(reversed(parts))

    def visit_Attribute(self, node):
        chain = self.resolve_attribute_chain(node)

        if chain is None:
            self.generic_visit(node)
            return

        base_name = chain[0]

        if base_name in self.imports:
            obj = self.imports[base_name]

            # Traverse remaining attributes
            for attr in chain[1:]:
                try:
                    if hasattr(obj, attr):
                        obj = getattr(obj, attr)
                    else:
                        self.errors.append({
                            "type": "attribute_error",
                            "object": base_name,
                            "attribute": attr,
                            "line": node.lineno
                        })
                        break
                except Exception:
                    # FIX: Avoid C-extension reflection failure
                    break

        self.generic_visit(node)

    # =========================
    # FUNCTION CALL CHECK
    # =========================

    def visit_Call(self, node):
        """
        # FIX: Removed inspect.signature logic
        Signature inspection is unstable for C-extensions.
        Now only checks existence of callable attribute.
        """

        if isinstance(node.func, ast.Attribute):
            chain = self.resolve_attribute_chain(node.func)

            if chain is not None:
                base_name = chain[0]

                if base_name in self.imports:
                    obj = self.imports[base_name]

                    for attr in chain[1:]:
                        try:
                            if hasattr(obj, attr):
                                obj = getattr(obj, attr)
                            else:
                                self.errors.append({
                                    "type": "attribute_error",
                                    "object": base_name,
                                    "attribute": attr,
                                    "line": node.lineno
                                })
                                break
                        except Exception:
                            break

        self.generic_visit(node)


# =========================
# ANALYSIS FUNCTION
# =========================

def analyze_library_api(code: str):
    result = {
        "libapi_analyzed": False,
        "name_error": 0,
        "attribute_error": 0,
        "module_not_found": 0,  # kept for schema consistency
        "total_libapi_errors": 0,
        "libapi_details": []
    }

    try:
        tree = ast.parse(code)
        visitor = LibraryAPIVistor()
        visitor.visit(tree)

        result["libapi_analyzed"] = True
        result["libapi_details"] = visitor.errors

        for err in visitor.errors:
            if err["type"] in result:
                result[err["type"]] += 1

        result["total_libapi_errors"] = len(visitor.errors)

    except Exception:
        pass

    return result


# DYNAMIC

- Input : we give the `row` of dataframe, code,task id and dataset type
```
DYNAMIC_ANALYSIS_INPUT_SCHEMA = {
    "row": pd.DataFrame,      # Single dataset row containing tests + metadata
    "dataset_type": str,      # "ds1000" | "humaneval" | "mbpp"
    "task_id": str,           # Unique task identifier
    "generated_code": str     # LLM-generated Python code
}
```
- Output : error infomration
```
DYNAMIC_ANALYSIS_OUTPUT_SCHEMA = {
    "status": str,             # "passed" | "failed"

    "error_type": str,         # RuntimeError | AssertionError |
                               # SyntaxError | Timeout |
                               # TestParseError | UnknownDataset |
                               # None (if passed)

    "error_message": str,      # Raw exception message

    "line_number": str | int,  # Extracted failing line if available

    "test_case": str,          # Test case that triggered failure

    "testcase_output": str,    # Captured stdout or traceback snippet

    "generated_code": str,     # Evaluated code snapshot

    "dataset": str,            # ds1000 | humaneval | mbpp

    "task_id": str             # Task identifier
}
```
- Main fucntion we call is `run_dynamic_driver_dynamic_analysis`

In [ ]:
TIMEOUT_SECONDS = 10


class TimeoutException(Exception):
    pass


def timeout_handler(signum, frame):
    raise TimeoutException("Execution exceeded timeout")


def extract_syntax_error_line(error_message: str) -> str:

    match = re.search(r'\(<string>,\s*line\s+(\d+)\)', error_message)
    if match:
        return match.group(1)
    return ""

#objects are stores as hasesh which can't be used thus we serialize this!
def serialize_value(value: Any, max_length: int = 500) -> str:
    try:
        if isinstance(value, pd.DataFrame):
            try:
                serialized = value.to_dict('list')
                result = f"DataFrame({serialized})"
            except:
                result = f"DataFrame:\n{value.to_string()}"
        elif isinstance(value, pd.Series):
            try:
                serialized = value.to_dict()
                result = f"Series({serialized})"
            except:
                result = f"Series:\n{value.to_string()}"
        elif isinstance(value, np.ndarray):
            try:
                result = f"array({value.tolist()})"
            except:
                result = f"array({repr(value)})"
        elif isinstance(value, (dict, list, tuple)):
            result = str(value)
        elif value is None:
            return "None"
        else:
            try:
                if pd.isna(value):
                    return "NaN"
            except (TypeError, ValueError):
                pass
            result = str(value)

        if len(result) > max_length:
            result = result[:max_length] + "...[truncated]"

        return result
    except Exception as e:
        return f"<Serialization Error: {str(e)}>"


def extract_ds1000_test_cases(generated_code: str, code_context: str) -> List[List[str]]:

    test_cases_data = []

    try:
        # testing environment
        test_env = {}
        exec(code_context, test_env)

        if 'generate_test_case' not in test_env:
            return []

        for test_id in range(1, 10):
            try:
                test_input, expected_result = test_env['generate_test_case'](test_id)

                exec_env = {}
                exec_env['test_input'] = test_input

                try:
                    exec(generated_code, exec_env)
                    actual_result = exec_env.get('result', '<No result variable>')
                except Exception as exec_error:
                    actual_result = f"<Execution Error: {str(exec_error)}>"

                # Serialize the values from objects
                input_str = serialize_value(test_input)
                expected_str = serialize_value(expected_result)
                actual_str = serialize_value(actual_result)

                test_cases_data.append([input_str, expected_str, actual_str])

            except Exception:
                break
    except Exception as e:
        pass
    return test_cases_data


def extract_humaneval_test_cases(generated_code: str, test_code: str, entry_point: str) -> List[List[str]]:
    test_cases_data = []
    try:
        tree = ast.parse(test_code)

        test_env = {}
        exec(generated_code, test_env)

        if entry_point not in test_env:
            return []

        func = test_env[entry_point]

        #using as to find all the assert statements!
        for node in ast.walk(tree):
            if isinstance(node, ast.Assert): #ast.Assert
                try:
                    test_node = node.test

                    # Handle assert func(input) == expected
                    if isinstance(test_node, ast.Compare): #ast.Compare
                        left = test_node.left
                        comparators = test_node.comparators
                        if isinstance(left, ast.Call):
                            args = []
                            for arg in left.args:
                                try:
                                    arg_value = ast.literal_eval(arg)
                                    args.append(arg_value)
                                except:
                                    args.append("<complex_arg>")

                            if comparators:
                                try:
                                    expected_value = ast.literal_eval(comparators[0])
                                except:
                                    expected_value = "<complex_expected>"
                            else:
                                expected_value = "<unknown>"

                            # Execute function with args to get actual
                            try:
                                actual_value = func(*args)
                            except Exception as exec_error:
                                actual_value = f"<Error: {str(exec_error)}>"

                            # Serialize
                            input_str = serialize_value(tuple(args) if len(args) > 1 else (args[0] if args else "()"))
                            expected_str = serialize_value(expected_value)
                            actual_str = serialize_value(actual_value)

                            test_cases_data.append([input_str, expected_str, actual_str])
                except Exception:
                    continue

    except Exception as e:
        pass

    return test_cases_data

"""MODIFIED!"""
def extract_mbpp_test_cases(generated_code: str, test_list: List[str], test_imports: List[str]) -> List[List[str]]:
    import ast
    test_cases_data = []

    try:
        test_env = {}

        # Load imports
        for imp in test_imports:
            if imp.strip():
                exec(imp, test_env)

        # Load generated code
        exec(generated_code, test_env)

        for test_assertion in test_list:
            try:
                tree = ast.parse(test_assertion)

                for node in ast.walk(tree):
                    if isinstance(node, ast.Assert):
                        test_node = node.test

                        if isinstance(test_node, ast.Compare):
                            left = test_node.left
                            comparators = test_node.comparators

                            if isinstance(left, ast.Call):

                                func_name = left.func.id
                                func = test_env.get(func_name)

                                # Extract input args
                                args = []
                                for arg in left.args:
                                    try:
                                        arg_value = eval(
                                            compile(ast.Expression(arg), "<string>", "eval"),
                                            test_env
                                        )
                                        args.append(arg_value)
                                    except:
                                        args.append("<complex_arg>")

                                # Extract expected value
                                try:
                                    expected_value = eval(
                                        compile(ast.Expression(comparators[0]), "<string>", "eval"),
                                        test_env
                                    )
                                except:
                                    expected_value = "<complex_expected>"

                                # Execute function
                                try:
                                    actual_value = func(*args)
                                except Exception as exec_error:
                                    actual_value = f"<Error: {str(exec_error)}>"

                                input_str = serialize_value(args[0] if len(args)==1 else tuple(args))
                                expected_str = serialize_value(expected_value)
                                actual_str = serialize_value(actual_value)

                                test_cases_data.append([input_str, expected_str, actual_str])

            except Exception:
                continue

    except Exception:
        pass

    return test_cases_data


def execute_with_timeout(func, args, timeout=TIMEOUT_SECONDS):

    result_container = {"result": None, "exception": None, "traceback": None}

    def wrapper():
        try:
            result_container["result"] = func(*args)
        except Exception as e:
            result_container["exception"] = e
            result_container["traceback"] = traceback.format_exc()

    thread = threading.Thread(target=wrapper)
    thread.daemon = True
    thread.start()
    thread.join(timeout=timeout)

    if thread.is_alive():
        # Thread still running - timeout occurred
        # Extract generated_code from args if available
        gen_code = args[0] if args else ""
        return {
            "status": "failed",
            "error_type": "TimeoutError",
            "error_message": "Execution exceeded timeout (likely infinite loop or recursion)",
            "line_number": "",
            "test_case": "",
            "testcase_output": "",
            "generated_code": gen_code
        }

    if result_container["exception"] is not None:
        e = result_container["exception"]
        is_assertion_error = isinstance(e, AssertionError)
        is_syntax_error = isinstance(e, SyntaxError)
        tb = traceback.extract_tb(e.__traceback__)
        full_traceback = result_container["traceback"] or ""


        if is_assertion_error:
            line_num = ""
        elif is_syntax_error:
            line_num = extract_syntax_error_line(str(e))
        else:
            string_frames = [frame for frame in tb if '<string>' in frame.filename]
            line_num = min((frame.lineno for frame in string_frames), default="") if string_frames else ""

        gen_code = args[0] if args else ""

        return {
            "status": "failed",
            "error_type": type(e).__name__,
            "error_message": str(e),
            "line_number": str(line_num) if line_num else "",
            "test_case": "",
            "testcase_output": full_traceback if is_assertion_error else "",
            "generated_code": gen_code
        }

    if result_container["result"] is not None:
        return result_container["result"]

    gen_code = args[0] if args else ""
    return {
        "status": "failed",
        "error_type": "UnknownError",
        "error_message": "Execution completed but no result returned",
        "line_number": "",
        "test_case": "",
        "testcase_output": "",
        "generated_code": gen_code
    }


def execute_ds1000_test_inner(generated_code: str, code_context: str) -> Dict[str, Any]:
    """
    Inner function to execute DS1000 test (runs inside timeout wrapper).

    Args:
        generated_code: Generated code snippet
        code_context: Test context with test_execution function

    Returns:
        Dictionary with test results including test_case and testcase_output
    """
    test_env = {}
    line_offset = 0

    try:

        exec(code_context, test_env)

        # Compute line offset from exec_context template
        # DS1000's test_execution() wraps generated_code inside exec_context,
        # prepending setup lines before [insert]. Traceback line numbers refer
        # to the combined code, so we must subtract the offset to map back to
        # the original generated_code.
        exec_ctx = test_env.get('exec_context', '')
        if exec_ctx and '[insert]' in exec_ctx:
            line_offset = exec_ctx.split('[insert]')[0].count('\n')

        # Execute the test
        test_env['test_execution'](generated_code)

        return {
            "status": "passed",
            "error_type": "",
            "error_message": "",
            "line_number": "",
            "test_case": "",
            "testcase_output": "",
            "generated_code": generated_code
        }

    except Exception as e:
        is_assertion_error = isinstance(e, AssertionError)
        is_syntax_error = isinstance(e, SyntaxError)
        tb = traceback.extract_tb(e.__traceback__)
        full_traceback = traceback.format_exc()

        # Get line number based on error type
        if is_assertion_error:
            # AssertionErrors don't populate line_number
            line_num = ""
        elif is_syntax_error:
            # Extract line number from SyntaxError message
            line_num = extract_syntax_error_line(str(e))
            # Adjust for exec_context offset
            if line_num and line_offset:
                line_num = str(max(1, int(line_num) - line_offset))
        else:
            # For runtime errors, use the last <string> frame (innermost exec
            # context = actual error location), then adjust for the offset
            string_frames = [frame for frame in tb if '<string>' in frame.filename]
            if string_frames:
                raw_line = string_frames[-1].lineno
                line_num = str(max(1, raw_line - line_offset)) if raw_line else ""
            else:
                line_num = ""

        # Extract test case data for all failed tests
        test_case_data = extract_ds1000_test_cases(generated_code, code_context)
        test_case_json = json.dumps(test_case_data) if test_case_data else ""

        return {
            "status": "failed",
            "error_type": type(e).__name__,
            "error_message": str(e),
            "line_number": str(line_num) if line_num else "",
            "test_case": test_case_json,
            "testcase_output": full_traceback if is_assertion_error else "",
            "generated_code": generated_code
        }


def execute_ds1000_test(generated_code: str, code_context: str) -> Dict[str, Any]:
    """
    Execute DS1000 test with timeout protection.

    Args:
        generated_code: Generated code snippet
        code_context: Test context with test_execution function

    Returns:
        Dictionary with test results
    """
    return execute_with_timeout(execute_ds1000_test_inner, (generated_code, code_context))


def execute_humaneval_test_inner(generated_code: str, test_code: str, entry_point: str) -> Dict[str, Any]:
    """
    Inner function to execute HumanEval test (runs inside timeout wrapper).

    Args:
        generated_code: Generated function code
        test_code: Test code with check() function
        entry_point: Function name to test

    Returns:
        Dictionary with test results including test_case and testcase_output
    """
    test_env = {}

    try:

        exec(generated_code, test_env)

        exec(test_code, test_env)

        if entry_point in test_env and 'check' in test_env:
            test_env['check'](test_env[entry_point])
        else:
            raise NameError(f"Entry point '{entry_point}' or 'check' function not found")

        return {
            "status": "passed",
            "error_type": "",
            "error_message": "",
            "line_number": "",
            "test_case": "",
            "testcase_output": "",
            "generated_code": generated_code
        }

    except Exception as e:
        is_assertion_error = isinstance(e, AssertionError)
        is_syntax_error = isinstance(e, SyntaxError)
        tb = traceback.extract_tb(e.__traceback__)
        full_traceback = traceback.format_exc()

        if is_assertion_error:
            line_num = ""
        elif is_syntax_error:
            line_num = extract_syntax_error_line(str(e))
        else:

            string_frames = [frame for frame in tb if '<string>' in frame.filename]
            line_num = min((frame.lineno for frame in string_frames), default="") if string_frames else ""

        # Extract test case data for all failed tests
        test_case_data = extract_humaneval_test_cases(generated_code, test_code, entry_point)
        test_case_json = json.dumps(test_case_data) if test_case_data else ""

        return {
            "status": "failed",
            "error_type": type(e).__name__,
            "error_message": str(e),
            "line_number": str(line_num) if line_num else "",
            "test_case": test_case_json,
            "testcase_output": full_traceback if is_assertion_error else "",
            "generated_code": generated_code
        }


def execute_humaneval_test(generated_code: str, test_code: str, entry_point: str) -> Dict[str, Any]:
    """
    Execute HumanEval test with timeout protection.

    Args:
        generated_code: Generated function code
        test_code: Test code with check() function
        entry_point: Function name to test

    Returns:
        Dictionary with test results
    """
    return execute_with_timeout(execute_humaneval_test_inner, (generated_code, test_code, entry_point))


def execute_mbpp_test_inner(generated_code: str, test_list: List[str], test_imports: List[str]) -> Dict[str, Any]:

    test_env = {}

    try:
        # Run imports
        for imp in test_imports:
            if imp.strip():
                exec(imp, test_env)

        # Run generated code
        exec(generated_code, test_env)

        # Run assertions
        for test_assertion in test_list:
            if test_assertion.strip():
                exec(test_assertion, test_env)

        return {
            "status": "passed",
            "error_type": "",
            "error_message": "",
            "line_number": "",
            "test_case": "",
            "testcase_output": "",
            "generated_code": generated_code
        }

    except Exception as e:
        full_traceback = traceback.format_exc()
        #modified!
        test_case_data = extract_mbpp_test_cases(generated_code, test_list, test_imports)
        test_case_json = json.dumps(test_case_data) if test_case_data else ""
        return {
            "status": "failed",
            "error_type": type(e).__name__,
            "error_message": str(e),
            "line_number": "",
            "test_case": test_case_json,
            "testcase_output": full_traceback,
            "generated_code": generated_code
        }


def execute_mbpp_test(generated_code: str, test_list: List[str], test_imports: List[str]) -> Dict[str, Any]:
    """
    Execute MBPP test with timeout protection.

    Args:
        generated_code: Generated function code
        test_list: List of test assertions
        test_imports: List of import statements

    Returns:
        Dictionary with test results
    """
    return execute_with_timeout(execute_mbpp_test_inner, (generated_code, test_list, test_imports))


def update_syntax_error_line_numbers(csv_path: Path) -> int:
    """
    Post-process existing CSV to extract line numbers from SyntaxError messages.
    This ensures any SyntaxErrors that slipped through without line numbers get updated.

    Args:
        csv_path: Path to the results CSV file

    Returns:
        Number of rows updated
    """
    print("\nPost-processing: Updating SyntaxError line numbers...")

    try:
        df = pd.read_csv(csv_path)
        updates = 0

        # Find SyntaxErrors with empty line_number
        for idx, row in df.iterrows():
            if row['error_type'] == 'SyntaxError' and pd.notna(row['error_message']):
                # Check if line_number is empty or NaN
                if pd.isna(row['line_number']) or str(row['line_number']).strip() == '':
                    # Extract line number from error message
                    line_num = extract_syntax_error_line(str(row['error_message']))
                    if line_num:
                        df.at[idx, 'line_number'] = line_num
                        updates += 1

        if updates > 0:
            df.to_csv(csv_path, index=False)
            print(f"✓ Updated {updates} SyntaxError entries with line numbers")
        else:
            print("✓ All SyntaxError entries already have line numbers")

        return updates

    except Exception as e:
        print(f"✗ Failed to update SyntaxError line numbers: {e}")
        traceback.print_exc()
        return 0


In [ ]:
def run_dynamic_driver_dynamic_analysis(
    row: pd.DataFrame,
    dataset_type: str,
    task_id: str,
    generated_code: str = "generated_code"
) -> pd.DataFrame:
    """
    Dynamic execution driver.

    Args:
        df: DataFrame containing generated code + test columns
        dataset_type: "DS1000", "HumanEval", or "MBPP"
        code_column: column containing code to evaluate

    Returns:
        DataFrame with structured dynamic execution results
    """

    results = []


    if dataset_type == "ds1000":
        code_context = str(row.get("code_context", ""))
        result = execute_ds1000_test(generated_code, code_context)

    elif dataset_type == "humaneval":
        test_code = str(row.get("test", ""))
        entry_point = str(row.get("entry_point", ""))
        result = execute_humaneval_test(generated_code, test_code, entry_point)

    elif dataset_type == "mbpp":
        try:
            generated_code = str(generated_code).strip()

            raw_test_list = str(row.get("test_list", ""))
            test_list = re.findall(r"'([^']*)'", raw_test_list)

            raw_test_imports = str(row.get("test_imports", ""))
            test_imports = re.findall(r"'([^']*)'", raw_test_imports)

            result = execute_mbpp_test(
                generated_code,
                test_list,
                test_imports
            )

        except Exception as e:
            result = {
                "status": "failed",
                "error_type": "TestParseError",
                "error_message": str(e),
                "line_number": "",
                "test_case": "",
                "testcase_output": "",
                "generated_code": generated_code
            }

    else:
        result = {
            "status": "failed",
            "error_type": "UnknownDataset",
            "error_message": f"Unsupported dataset: {dataset_type}",
            "line_number": "",
            "test_case": "",
            "testcase_output": "",
            "generated_code": generated_code
        }

        result["dataset"] = dataset_type
        result["task_id"] = task_id



    return result

# KG

# FAULT INFORMATION

- Input is all the error info and we just group it here and decide the status.
```
DETECTION_STAGE_OUTPUT_SCHEMA = {
    "dataset": str,
    "task_id": str,

    "ast": dict,
    "lib": dict,
    "dynamic": dict
}
```
- The function we will call is `build_fault_information`
- the output schema looks like,
```
fault_information = {
        "dataset" ,
        "status"  ,
        "task_id" ,
        "ast_info" ,
        "lib_info" ,
        "dynamic_info"
    }
```

In [ ]:
def build_fault_information(
    dataset: str,
    task_id: str,
    ast_result: Dict[str, Any],
    lib_result: Dict[str, Any] = None,
    dynamic_result: Dict[str, Any] = None
) -> Dict[str, Any]:
    """
    Build structured fault_information dictionary
    compatible with generate_patch_driver.
    """

    ast_has_error = bool(ast_result.get("ast_errors"))

    lib_has_error = False
    if lib_result:
        lib_has_error = lib_result.get("total_libapi_errors", 0) > 0 #damm one line,prevented it from patch.

    dynamic_has_error = False
    if dynamic_result:
        dynamic_has_error = dynamic_result.get("status") == "failed"

    if ast_has_error or lib_has_error or dynamic_has_error:
        status = "hallucinated"
    else:
        status = "passed"

    fault_information = {
        "dataset": dataset,
        "status": status,
        "task_id": task_id,

        # Store raw dicts (NOT JSON strings)
        "ast_info": ast_result if ast_has_error else None,
        "lib_info": lib_result if lib_has_error else None,
        "dynamic_info": dynamic_result if dynamic_has_error else None
    }

    return fault_information

# PATCH GENERATION

In [ ]:
def extract_ast_errors(ast_info: Dict) -> List[Tuple[int, int, str, str]]:
    """
    Returns:
        (start_line, end_line, error_type, message)
    """

    if not ast_info or "ast_errors" not in ast_info:
        return []

    errors = []

    for item in ast_info["ast_errors"]:
        start = item.get("start_line")
        end = item.get("end_line", start)
        etype = item.get("type", "AST_Error")
        message = item.get("message", "")

        if start:
            errors.append((int(start), int(end), etype, message))

    return errors

from typing import List, Tuple, Dict


def extract_lib_errors(lib_info: Dict) -> List[Tuple[int, int, str, str]]:
    """
    Extract LIB_API errors from structured lib_info dict.

    Expected schema:
    {
        "libapi_details": [
            {
                "type": "attribute_error",
                "object": "math",
                "attribute": "square",
                "line": 4
            }
        ]
    }

    Returns:
        List of (start_line, end_line, error_type, message)
    """

    if not lib_info:
        return []

    details = lib_info.get("libapi_details", [])
    if not isinstance(details, list):
        return []

    errors = []

    for item in details:
        if not isinstance(item, dict):
            continue

        line = item.get("line")
        err_type = item.get("type", "lib_error")

        if not line:
            continue

        # Construct meaningful message for patch model
        if err_type == "attribute_error":
            obj = item.get("object", "")
            attr = item.get("attribute", "")
            message = f"Attribute '{attr}' not found in '{obj}'"
        elif err_type == "name_error":
            name = item.get("name", "")
            message = f"Name '{name}' not found in module"
        else:
            message = "Library API error"

        errors.append(
            (int(line), int(line), f"lib:{err_type}", message)
        )

    return errors
from typing import List, Tuple, Dict


def extract_dynamic_errors(dynamic_info: Dict) -> List[Tuple[int, int, str, str]]:
    """
    Extract runtime-based dynamic errors for patching.

    Only patches errors that:
        - status == failed
        - have valid line_number
        - are NOT logical assertion failures

    Returns:
        List of (start_line, end_line, error_type, message)
    """

    if not dynamic_info:
        return []

    if dynamic_info.get("status") != "failed":
        return []

    error_type = dynamic_info.get("error_type", "")
    error_message = dynamic_info.get("error_message", "")
    line_number = dynamic_info.get("line_number")

    # ❌ Do NOT patch logical test failures
    if error_type in ["AssertionError", "WrongAnswer", "Timeout"]:
        return []

    if not line_number:
        return []

    try:
        line_num = int(float(str(line_number).strip()))
        if line_num <= 0:
            return []
    except (ValueError, TypeError):
        return []

    return [(line_num, line_num, error_type, error_message)]
from typing import List, Tuple, Optional


def generate_full_patch(
    code: str,
    errors: List[Tuple[int, int, str, str]],
    source_name: str = "ast"
) -> Optional[str]:
    """
    Generic full patch generator.

    Args:
        code: Original code
        errors: List of (start_line, end_line, error_type, message)
        source_name: error source label (ast, lib, dynamic, etc.)

    Returns:
        Patched full code
    """

    if not code:
        return None

    lines = code.split("\n")
    total_lines = len(lines)

    start_markers = {}
    end_markers = {}

    for start, end, etype, message in errors:

        # Validate line numbers
        if not start or start < 1 or start > total_lines:
            continue

        if not end or end < start:
            end = start

        if end > total_lines:
            end = total_lines

        # Construct label
        if message:
            label = f"{source_name}: {etype}" # | {message}"
        else:
            label = f"{source_name}: {etype}"

        start_markers.setdefault(start - 1, []).append(label)
        end_markers.setdefault(end - 1, []).append(label)

    # No valid errors
    if not start_markers:
        return code

    patched_lines = []

    for i, line in enumerate(lines):

        # Insert START markers
        if i in start_markers:
            for label in start_markers[i]:
                patched_lines.append(
                    f"<<<< [ERROR START] ({label})"
                )

        patched_lines.append(line)

        # Insert END markers
        if i in end_markers:
            for label in end_markers[i]:
                patched_lines.append(
                    f"[ERROR END] ({label}) >>>>"
                )

    return "\n".join(patched_lines)

### generate_patch_driver

In [ ]:
def generate_patch_driver(
    fault_information: Dict,
    generated_code: str
) -> Optional[Dict]:

    if not generated_code:
        return None

    all_errors: List[Tuple[int, int, str, str]] = []
    error_sources = []


    ast_info = fault_information.get("ast_info")

    if ast_info:
        ast_errors = extract_ast_errors(ast_info)

        if ast_errors:
            all_errors.extend(ast_errors)
            error_sources.append("ast")

            patched_code = generate_full_patch(
                code=generated_code,
                errors=all_errors,
                source_name="ast"
            )

            return {
                "dataset": fault_information.get("dataset"),
                "status": fault_information.get("status"),
                "task_id": fault_information.get("task_id"),
                "generated_code": generated_code,
                "patched_code": patched_code,
                "error_sources": "ast",
                "error_types": ",".join(e[2] for e in all_errors),
                "error_lines": ",".join(f"{e[0]}-{e[1]}" for e in all_errors)
            }

    dynamic_info = fault_information.get("dynamic_info")

    if dynamic_info:
        dynamic_errors = extract_dynamic_errors(dynamic_info)

        if dynamic_errors:
            all_errors.extend(dynamic_errors)
            error_sources.append("dynamic")


    lib_info = fault_information.get("lib_info")

    if lib_info:
        lib_errors = extract_lib_errors(lib_info)
        if lib_errors:
            all_errors.extend(lib_errors)
            error_sources.append("lib")

    if not all_errors:
        return {
            "dataset": fault_information.get("dataset"),
            "status": fault_information.get("status"),
            "task_id": fault_information.get("task_id"),
            "generated_code": generated_code,
            "patched_code": generated_code,
            "error_sources": "",
            "error_types": "",
            "error_lines": ""
        }

    patched_code = generate_full_patch(
        code=generated_code,
        errors=all_errors,
        source_name=",".join(error_sources)
    )

    return {
        "dataset": fault_information.get("dataset"),
        "status": fault_information.get("status"),
        "task_id": fault_information.get("task_id"),
        "generated_code": generated_code,
        "patched_code": patched_code,
        "error_sources": ",".join(error_sources),
        "error_types": ",".join(e[2] for e in all_errors),
        "error_lines": ",".join(f"{e[0]}-{e[1]}" for e in all_errors)
    }

# MODEL LOADING

In [ ]:

model_id = "Qwen/Qwen2.5-Coder-3B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    low_cpu_mem_usage=True
)

print(f"Success! {model_id} loaded using standard transformers.")

# CODE GEN HELPERS

### HumanEval

In [ ]:

def construct_prompt_humaneval(docstring_prompt):

    system_message = (
        "You are an expert Python developer. Your task is to complete the function "
        "provided by the user. Follow the docstring exactly. "
        "Provide your output ONLY as a single Python code block starting with ```python."
    )

    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": f"Complete this Python function:\n{docstring_prompt}"}
    ]

    return messages

#------------------------------------------------------------------------------------------------------------

def run_test_humaneval(generated_code, test_script, entry_point):

    execution_code = f"{generated_code}\n\n{test_script}\ncheck({entry_point})"

    try:
        exec_context = {}
        exec(execution_code, exec_context)
        return "passed"

    except AssertionError as e:
        return "Failed: Logic Hallucination (Assertion Error)"
    except SyntaxError as e:
        return f"Failed: Syntax Hallucination ({e})"
    except Exception as e:
        return f"Failed: {type(e).__name__}: {str(e)}"

#run_test(clean_code,t,df['entry_point'][9])

def extract_python_code_humaneval(text):

    pattern = r"```(?:python)?\n?(.*?)```"
    match = re.search(pattern, text, re.DOTALL)

    if match:
        return match.group(1).strip()
    return text.strip()


# clean_code = extract_python_code(response)
# exec(clean_code)
# print(clean_code)

#------------------------------------------------------------------------------------------------------------



### MBPP


In [ ]:
def construct_prompt_mbpp(prompt_text, signature):

    system_message = (
        "You are an expert Python developer. Your task is to implement a function "
        "based on a description and a specific function signature. "
        "Provide your output ONLY as a single Python code block starting with ```python."
    )


    user_content = (
        f"Problem Description:\n{prompt_text}\n\n"
        f"Please implement this exact function:\n{signature}"
    )

    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": user_content}
    ]

    return messages
#print(construct_prompt(mbpp_df['prompt'][0],mbpp_df['function_signature'][0]))

def extract_python_code_mbpp(text):

    pattern = r"```(?:python)?\n?(.*?)```"
    match = re.search(pattern, text, re.DOTALL)

    if match:
        return match.group(1).strip()
    return text.strip()


#clean_code = extract_python_code(response)
def verify_code_mbpp(generated_code, test_list):
    namespace = {}
    try:
        exec(generated_code, namespace)
        for test_case in test_list:
            exec(test_case, namespace)
        return "Passed"
    except AssertionError as e:
        return "Failed: Logic Hallucination (Assertion Error)"
    except SyntaxError as e:
        return f"Failed: Syntax Hallucination ({e})"
    except Exception as e:
        return f"Failed: {type(e).__name__}: {str(e)}"

### DS1000

In [ ]:
def run_ds1000_official_test_ds1k(clean_code, code_context):
    """
    Executes the code_context to load the official test_execution function,
    then runs the generated code through it.
    """
    # Create a sandbox environment
    test_env = {}

    try:

        exec(code_context, test_env)
        test_env['test_execution'](clean_code)

        return "passed"

    except AssertionError:
        return "Failed: Logic Error (Assertion failed)"
    except Exception as e:
        # Catch syntax errors in model code or execution issues
        return f"Failed: {type(e).__name__}: {str(e)}"

In [ ]:
def extract_only_exec_context_wi(code_context):
    """
    Specifically extracts the raw string assigned to exec_context.
    This contains only the imports and variable mappings used in the test.
    Along with the insert marker.
    """
    # Look for the raw string pattern: exec_context = r""" ... """
    pattern = r'exec_context = r"""(.*?)"""'
    match = re.search(pattern, code_context, re.DOTALL)

    if match:
        content = match.group(1).strip()
        # We leave the [insert] marker out of the prompt snippet
        # so the model doesn't get confused by the tag itself.
        return content.strip()

    return "import pandas as pd\nimport numpy as np" # Basic fallback

In [ ]:
def contruct_prompt_ds1k_v4(raw_prompt, exec_context_snippet):
    """
    Simple DS-1000 prompt: minimal code, minimal hallucination.
    """

    system_message = (
        "You are a Python data scientist.\n"
        "You write concise, correct Python code for data manipulation.\n"
        "You prefer direct, vectorized solutions over complex logic."
    )

    user_message = (
        "You are given a Python code snippet with a placeholder [insert].\n"
        "Your code will be INSERTED at that position.\n\n"

        "===== EXISTING CODE =====\n"
        f"{exec_context_snippet}\n\n"

        "===== TASK =====\n"
        f"{raw_prompt}\n\n"

        "===== GUIDELINES =====\n"
        "- Think deeply about the given TASK before coding.\n"
        "- Write the simplest correct solution.\n"
        "- Prefer short, direct Pandas / NumPy operations.\n"
        "- Do NOT define helper functions or classes.\n"
        "- Do NOT print anything.\n"
        "- You MAY add imports if needed.\n"
        "- Use existing variables from the context.\n"
        "- Make sure `result` variable is declared before using it.\n"
        "- Give proper intendation at [INSERT] if the line before [INSERT] is a function.\n"
        "- Ensure the final output is available in variable `result`.This is really important.\n\n"

        "===== OUTPUT =====\n"
        "Return ONLY raw Python code.\n"
    )

    return [
        {"role": "system", "content": system_message},
        {"role": "user", "content": user_message}
    ]


# HALLUINCATION PIPELINE

#### run_full_halluincation_pipeline

In [ ]:
def run_full_hallucination_pipeline(
    row,
    dataset_type: str,
    task_id: str,
    code: str
):
    """
    Full staged hallucination detection pipeline.

    Order:
        1. AST
        2. Dynamic
        3. LIB_API (only if dynamic fails)
        4. Patch generation
    """

    ast_result = analyze_ast_for_patch(code)

    if ast_result["ast_errors"]:
        fault_information = build_fault_information(
            dataset=dataset_type,
            task_id=task_id,
            ast_result=ast_result,
            lib_result=None,
            dynamic_result=None,
        )
        out = generate_patch_driver(
            fault_information=fault_information,
            generated_code=code
        )
        out["fault_information"] = fault_information
        out["patch"] = out["patched_code"]
        return out

    dynamic_result = run_dynamic_driver_dynamic_analysis(
        row=row,
        dataset_type=dataset_type,
        task_id=task_id,
        generated_code=code
    )

    if dynamic_result.get("status") == "failed":

        lib_result = analyze_library_api(code)

        fault_information = build_fault_information(
            dataset=dataset_type,
            task_id=task_id,
            ast_result=ast_result,
            lib_result=lib_result,
            dynamic_result=dynamic_result,
        )
        out = generate_patch_driver(
            fault_information=fault_information,
            generated_code=code
        )
        out["fault_information"] = fault_information
        out["patch"] = out["patched_code"]
        return out

    fault_information = build_fault_information(
        dataset=dataset_type,
        task_id=task_id,
        ast_result=ast_result,
        lib_result=None,
        dynamic_result=None,
    )
    out = generate_patch_driver(
        fault_information=fault_information,
        generated_code=code
    )
    out["fault_information"] = fault_information
    out["patch"] = out["patched_code"]
    return out

In [ ]:
#works for dataset driven
def run_dataset_pipeline(df: pd.DataFrame, dataset_type: str):

    results = []

    for idx, row in df.iterrows():

        task_id = str(row.get("task_id", idx))

        # Select correct code column
        if dataset_type == "ds1000":
            code = str(row.get("full_code", ""))

        elif dataset_type == "humaneval":
            code = str(row.get("GENERATED_CODE", ""))

        elif dataset_type == "mbpp":
            code = str(row.get("GENERATED_CODE", ""))

        else:
            continue

        pipeline_output = run_full_hallucination_pipeline(
            row=row,
            dataset_type=dataset_type,
            task_id=task_id,
            code=code
        )

        results.append(pipeline_output)

        if idx % 100 == 0:
            print(f"[{dataset_type}] Processed {idx}/{len(df)}")

    return pd.DataFrame(results)

# APR

In [ ]:
# ---------- APR: One prompt per error type ----------

def _normalize_error_type(s: str) -> str:
    t = s.split(":")[-1].strip() if ":" in s else s.strip()
    return t.replace(" ", "")

def get_repair_category(error_types: str, fault_information: dict) -> str:
    """
    Classify error for repair. Returns: syntax, attribute, type, name, key, assertion, timeout, other, or skip (empty).
    """
    if not error_types or not str(error_types).strip():
        return "skip"
    parts = [p.strip() for p in str(error_types).split(",")]
    for p in parts:
        t = _normalize_error_type(p)
        if t in ("SyntaxError", "IndentationError"):
            return "syntax"
        if t in ("AttributeError", "attribute_error"):
            return "attribute"
        if t == "TypeError":
            return "type"
        if t in ("NameError", "name_error"):
            return "name"
        if t == "KeyError":
            return "key"
        if t in ("AssertionError", "WrongAnswer"):
            return "assertion"
        if t == "TimeoutError":
            return "timeout"
    return "other"


def build_prompt_syntax(patched_code: str, fault_information: dict, original_question: str, suggestions=None) -> list:
    """
    Build chat messages for syntax-only repair. Uses error message and line from ast_info or dynamic_info.
    """
    ast_info = fault_information.get("ast_info") or {}
    dynamic_info = fault_information.get("dynamic_info") or {}
    error_message = ""
    line_info = ""

    if ast_info and ast_info.get("ast_errors"):
        err = ast_info["ast_errors"][0]
        error_message = err.get("message", "Syntax or structural error")
        line_info = f"Line(s) {err.get('start_line', '?')}-{err.get('end_line', err.get('start_line', '?'))}"
    elif dynamic_info and dynamic_info.get("status") == "failed":
        error_message = dynamic_info.get("error_message", "Syntax or runtime error")
        line_info = f"Line {dynamic_info.get('line_number', '?')}" if dynamic_info.get("line_number") else ""

    system_message = (
        "You are an expert Python developer. Your ONLY task is to fix syntax or indentation errors.\n"
        "You must NOT change program logic, add new features, or refactor.\n"
        "Fix ONLY the reported line(s). Remove any ERROR markers from the output.\n"
        "Return ONLY a single Python code block wrapped in ```python and ```. No explanations."
    )
    user_content = (
        f"Original task or context:\n{original_question[:1500]}\n\n"
        f"Error: {error_message}\n{line_info}\n\n"
        "Buggy code (with optional error markers):\n"
        f"```\n{patched_code}\n```\n\n"
        "Fix the syntax/indentation only. Output the complete corrected code in one ```python block."
    )
    return [
        {"role": "system", "content": system_message},
        {"role": "user", "content": user_content}
    ]


def _parse_dynamic_info(fault_information: dict) -> dict:
    """Internal: parse dynamic_info from fault_information. Used only to extract fields."""
    d = fault_information.get("dynamic_info")
    if d is None:
        return {}
    if isinstance(d, dict):
        return d
    try:
        return json.loads(d) if isinstance(d, str) else {}
    except Exception:
        return {}


def extract_error_info_for_prompt(fault_information: dict) -> dict:
    """
    Extract minimal error info for prompts: error_type, error_message, line_number only.
    Avoids passing full dynamic_info (test_case, testcase_output, generated_code, etc.) to reduce tokens and hallucinations.
    """
    dyn = _parse_dynamic_info(fault_information or {})
    return {
        "error_type": dyn.get("error_type", ""),
        "error_message": dyn.get("error_message", ""),
        "line_number": dyn.get("line_number", ""),
    }


def extract_failing_test_cases_for_prompt(fault_information: dict, max_cases: int = 10) -> list:
    """
    Parse test_case from dynamic_info. Format: list of [input_str, expected_str, actual_str] per test.
    Filter to FAILING cases only (actual != expected). Return formatted strings:
    "For this INPUT {input} we get output {actual} but we need this {expected}"
    """
    dyn = _parse_dynamic_info(fault_information or {})
    raw = dyn.get("test_case", "")
    if not raw:
        return []
    if isinstance(raw, str) and raw.strip().startswith("["):
        try:
            raw = json.loads(raw)
        except Exception:
            return []
    if not isinstance(raw, list):
        return []
    out = []
    for item in raw[:max_cases]:
        if not isinstance(item, (list, tuple)) or len(item) < 3:
            continue
        input_str, expected_str, actual_str = str(item[0]), str(item[1]), str(item[2])
        if expected_str == actual_str:
            continue  # passed case, skip
        out.append(f"For this INPUT {input_str} we get output {actual_str} but we need this {expected_str}")
    return out


def build_prompt_attribute(patched_code: str, fault_information: dict, original_question: str, suggestions=None) -> list:
    err_info = extract_error_info_for_prompt(fault_information)
    err_msg = err_info.get("error_message") or "Attribute error"
    system = "You are a Python expert. Fix ONLY the attribute/API error. Replace wrong or missing attribute with the correct one. Return ONLY one ```python code block. No explanations."
    user = f"Task:\n{original_question[:1200]}\n\nError: {err_msg}\n\nBuggy code:\n```\n{patched_code}\n```\n\nFix the attribute error. Output full corrected code in one \`\`\`python block. Remove any ERROR markers."
    return [{"role": "system", "content": system}, {"role": "user", "content": user}]


def build_prompt_type(patched_code: str, fault_information: dict, original_question: str, suggestions=None) -> list:
    err_info = extract_error_info_for_prompt(fault_information)
    err_msg = err_info.get("error_message") or "Type error"
    system = "You are a Python expert. Fix ONLY the type/signature error (e.g. wrong keyword, wrong argument type). Return ONLY one ```python code block. No explanations."
    user = f"Task:\n{original_question[:1200]}\n\nError: {err_msg}\n\nBuggy code:\n```\n{patched_code}\n```\n\nFix the type or call signature. Output full corrected code in one \`\`\`python block. Remove any ERROR markers."
    return [{"role": "system", "content": system}, {"role": "user", "content": user}]


def build_prompt_name(patched_code: str, fault_information: dict, original_question: str, suggestions=None) -> list:
    err_info = extract_error_info_for_prompt(fault_information)
    err_msg = err_info.get("error_message") or "Name not defined"
    system = "You are a Python expert. Fix ONLY the NameError: add the missing import or define the missing variable. Return ONLY one ```python code block. No explanations."
    user = f"Task:\n{original_question[:1200]}\n\nError: {err_msg}\n\nBuggy code:\n```\n{patched_code}\n```\n\nAdd missing import or definition. Output full corrected code in one \`\`\`python block. Remove any ERROR markers."
    return [{"role": "system", "content": system}, {"role": "user", "content": user}]


def build_prompt_key(patched_code: str, fault_information: dict, original_question: str, suggestions=None) -> list:
    err_info = extract_error_info_for_prompt(fault_information)
    err_msg = err_info.get("error_message") or "Key error"
    system = "You are a Python expert. Fix ONLY the KeyError: use the correct key or handle missing key. Return ONLY one ```python code block. No explanations."
    user = f"Task:\n{original_question[:1200]}\n\nError: {err_msg}\n\nBuggy code:\n```\n{patched_code}\n```\n\nFix the key access. Output full corrected code in one \`\`\`python block. Remove any ERROR markers."
    return [{"role": "system", "content": system}, {"role": "user", "content": user}]


def build_prompt_assertion(patched_code: str, fault_information: dict, original_question: str, suggestions=None) -> list:
    failing = extract_failing_test_cases_for_prompt(fault_information)
    if not failing:
        test_str = "(no failing test details)"
    else:
        test_str = "\n".join(failing)
    system = "You are a Python expert. Fix the LOGIC so the tests pass. Do NOT change function signature or add helpers. Modify minimum lines. Return ONLY one ```python code block. No explanations."
    user = f"Task:\n{original_question[:1500]}\n\nBuggy code (fails tests):\n```\n{patched_code}\n```\n\nFailing tests (change logic accordingly):\n{test_str}\n\nFix the logic only. Output full corrected code in one \`\`\`python block. Remove any ERROR markers."
    return [{"role": "system", "content": system}, {"role": "user", "content": user}]


def build_prompt_timeout(patched_code: str, fault_information: dict, original_question: str, suggestions=None) -> list:
    err_info = extract_error_info_for_prompt(fault_information)
    err_msg = err_info.get("error_message") or "Timeout"
    system = "You are a Python expert. Fix the timeout: ensure loops terminate or reduce work (e.g. avoid brute force). Return ONLY one ```python code block. No explanations."
    user = f"Task:\n{original_question[:1200]}\n\nError: {err_msg}\n\nBuggy code:\n```\n{patched_code}\n```\n\nFix so it completes in time. Output full corrected code in one \`\`\`python block. Remove any ERROR markers."
    return [{"role": "system", "content": system}, {"role": "user", "content": user}]


def build_prompt_other(patched_code: str, fault_information: dict, original_question: str, suggestions=None) -> list:
    err_info = extract_error_info_for_prompt(fault_information)
    err_msg = err_info.get("error_message")
    if not err_msg:
        ast_info = fault_information.get("ast_info") or {}
        if ast_info.get("ast_errors"):
            err_msg = ast_info["ast_errors"][0].get("message", "Error")
    if not err_msg:
        err_msg = "Runtime or structural error"
    system = "You are a Python expert. Fix the error in the marked region or minimal lines. Return ONLY one ```python code block. No explanations."
    user = f"Task:\n{original_question[:1200]}\n\nError: {err_msg}\n\nBuggy code:\n```\n{patched_code}\n```\n\nFix the error. Output full corrected code in one \`\`\`python block. Remove any ERROR markers."
    return [{"role": "system", "content": system}, {"role": "user", "content": user}]


def build_apr_prompt(patched_code: str, fault_information: dict, original_question: str, error_types: str, suggestions=None) -> list:
    category = get_repair_category(error_types or "", fault_information or {})
    if category == "skip":
        return []
    builders = {
        "syntax": build_prompt_syntax,
        "attribute": build_prompt_attribute,
        "type": build_prompt_type,
        "name": build_prompt_name,
        "key": build_prompt_key,
        "assertion": build_prompt_assertion,
        "timeout": build_prompt_timeout,
        "other": build_prompt_other,
    }
    fn = builders.get(category, build_prompt_other)
    return fn(patched_code, fault_information or {}, original_question, suggestions)


def fix_code(
    generated_code: str,
    patch: str,
    fault_information: dict,
    row=None,
    error_types: str = None
) -> str:
    """
    Attempt repair using the prompt for the detected error category (syntax, attribute, type, name, key, assertion, timeout, other).
    """
    if not patch:
        patch = generated_code
    if error_types is None and fault_information:
        ast_info = fault_information.get("ast_info") or {}
        dynamic_info = fault_information.get("dynamic_info") or {}
        if isinstance(dynamic_info, str):
            try:
                dynamic_info = json.loads(dynamic_info) if dynamic_info else {}
            except Exception:
                dynamic_info = {}
        parts = []
        if ast_info and ast_info.get("ast_errors"):
            for e in ast_info["ast_errors"]:
                parts.append(e.get("type", "AST_Error"))
        if isinstance(dynamic_info, dict) and dynamic_info.get("status") == "failed":
            parts.append(dynamic_info.get("error_type", ""))
        error_types = ",".join(p for p in parts if p)

    category = get_repair_category(error_types or "", fault_information or {})
    if category == "skip":
        return generated_code

    original_question = ""
    if row is not None:
        original_question = str(row.get("prompt", row.get("prompt_2", "")))[:2000]

    messages = build_apr_prompt(patch, fault_information or {}, original_question, error_types or "", None)
    if not messages:
        return generated_code
    raw_response = generate_code(messages)
    fixed = extract_python_code_humaneval(raw_response)
    return fixed.strip() if (fixed and fixed.strip()) else generated_code

# DRIVER

### repair_with_max_passes

- Here `run_full_hallucination_pipeline` is ran 3 times.
- The error infomration is stored and
```

"final_status": "passed"/"failed",
"final_code": current_code,
"passes_used": pass_num,
"last_result": last_result
            
```

In [ ]:
def _summarize_fault_information(fault_information, max_len=500):
    """Extract concise error string from fault_information for display/CSV."""
    if not fault_information:
        return ""
    parts = []
    ast_info = fault_information.get("ast_info") or {}
    if isinstance(ast_info, str):
        try:
            ast_info = ast.literal_eval(ast_info) if ast_info else {}
        except Exception:
            ast_info = {}
    for e in (ast_info.get("ast_errors") or []):
        t = e.get("type", "") if isinstance(e, dict) else ""
        m = e.get("message", "") if isinstance(e, dict) else ""
        parts.append(f"{t}: {m}"[:200])
    dyn = fault_information.get("dynamic_info") or {}
    if isinstance(dyn, str):
        try:
            dyn = ast.literal_eval(dyn) if dyn else {}
        except Exception:
            dyn = {}
    if isinstance(dyn, dict) and dyn.get("status") == "failed":
        et = dyn.get("error_type", "")
        em = dyn.get("error_message", "")
        parts.append(f"{et}: {em}"[:200])
    s = " | ".join(parts) if parts else "unknown"
    return s[:max_len] if len(s) > max_len else s

def repair_with_max_passes(
    row,
    dataset_type: str,
    task_id: str,
    initial_code: str,
    max_passes: int = 3
):
    """
    Runs hallucination pipeline.
    If fails → applies fix_code()
    Repeats max 3 times.
    Returns ONLY final state (no full history).
    """

    current_code = initial_code
    last_result = None

    for pass_num in range(1, max_passes + 1):

        print(f"\n--- TRIAL {pass_num}/{max_passes} ---")
        code_snippet = current_code[:1500] + ("..." if len(current_code) > 1500 else "")
        print("CODE (input):\n", code_snippet)

        result = run_full_hallucination_pipeline(
            row=row,
            dataset_type=dataset_type,
            task_id=task_id,
            code=current_code
        )

        last_result = result

        err_summary = _summarize_fault_information(result.get("fault_information", {}))
        patch_snippet = (result.get("patch") or "")[:1500]
        if len(result.get("patch") or "") > 1500:
            patch_snippet += "..."
        print("ERROR:", err_summary or "(none)")
        print("PATCHED CODE:\n", patch_snippet)
        print("TRIAL COUNT:", pass_num)

        # ✅ If passed → stop early
        if result["status"] == "passed":
            print("✅ Code Passed All Checks")
            return {
                "final_status": "passed",
                "final_code": current_code,
                "passes_used": pass_num,
                "last_result": last_result
            }

        # ❌ If failed → attempt repair
        print("❌ Failure detected. Applying fix_code()")

        current_code = fix_code(
            generated_code=current_code,
            patch=result["patch"],
            fault_information=result["fault_information"],
            row=row,
            error_types=result.get("error_types")
        )

    # 🚨 After max passes exhausted
    print("❌ Failed After Max Passes")

    return {
        "final_status": "failed",
        "final_code": current_code,
        "passes_used": max_passes,
        "last_result": last_result
    }

### DATASET_CONFIG

In [ ]:
DATASET_CONFIG = {
    "humaneval": {
        "df": df_humaneval_f,
        "prompt_builder": lambda row: construct_prompt_humaneval(row["prompt"]),
        "code_extractor": extract_python_code_humaneval,
    },
    "mbpp": {
        "df": df_mbpp_f,
        "prompt_builder": lambda row: construct_prompt_mbpp(
            row["prompt"], row["function_signature"]
        ),
        "code_extractor": extract_python_code_humaneval,
    },
    "ds1000": {
        "df": df_ds1k_f,
        "prompt_builder": lambda row: contruct_prompt_ds1k_v4(
            row["prompt"],
            extract_only_exec_context_wi(row["code_context"])
        ),
        "code_extractor": extract_python_code_humaneval,
    }
}

### generate_code

In [ ]:
def generate_code(formatted_messages):
    inputs = tokenizer.apply_chat_template(
        formatted_messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt"
    ).to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=512,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    gen_ids = outputs[0][len(inputs["input_ids"][0]):]
    raw_response = tokenizer.decode(gen_ids, skip_special_tokens=True)

    return raw_response

### main_driver
- full dataset_config driver
- it calls the code generation and `repair_with_max_passes`

In [ ]:
def main_driver(max_passes=3, error_type_filter=None, save_csv_path=None):

    config_to_use = get_dataset_config_for_error_type(error_type_filter) if error_type_filter else None
    if config_to_use:
        print(f"Running for error_type_filter: {error_type_filter}")
        dataset_config = config_to_use
    else:
        dataset_config = DATASET_CONFIG

    all_results = []

    for dataset_name, config in dataset_config.items():

        df = config["df"]
        prompt_builder = config["prompt_builder"]
        code_extractor = config["code_extractor"]

        print(f"\n========== DATASET: {dataset_name} ==========\n")

        for idx, row in df.iterrows():

            task_id = row["task_id"]
            print(f"\nTask ID: {task_id}")

            # ---------------------------
            # 1️⃣ Generate Initial Code ONCE
            # ---------------------------
            formatted_messages = prompt_builder(row)
            raw_response = generate_code(formatted_messages)
            initial_code = code_extractor(raw_response)

            print("INITIAL CLEAN CODE:\n", initial_code)

            # ---------------------------
            # 2️⃣ Run Repair Loop
            # ---------------------------
            repair_result = repair_with_max_passes(
                row=row,
                dataset_type=dataset_name,
                task_id=task_id,
                initial_code=initial_code,
                max_passes=max_passes
            )

            # ---------------------------
            # 3️⃣ Collect result & print summary
            # ---------------------------
            last_result = repair_result.get("last_result", {})
            fault_info = last_result.get("fault_information", {})
            error_summary = _summarize_fault_information(fault_info)

            all_results.append({
                "dataset": dataset_name,
                "task_id": task_id,
                "initial_code": initial_code,
                "final_code": repair_result["final_code"],
                "final_status": repair_result["final_status"],
                "passes_used": repair_result["passes_used"],
                "error_summary": error_summary,
            })

            print("\n--- TASK SUMMARY ---")
            print("CODE (final):", (repair_result["final_code"] or "")[:800], "..." if len(repair_result["final_code"] or "") > 800 else "")
            print("ERROR:", error_summary or "(none)")
            print("PATCHED CODE (last):", (last_result.get("patch") or repair_result["final_code"] or "")[:800], "..." if len(last_result.get("patch") or repair_result["final_code"] or "") > 800 else "")
            print("TRIAL COUNT:", repair_result["passes_used"])

            if repair_result["final_status"] == "passed":
                print("✅ FINAL STATUS: PASSED")
            else:
                print("❌ FINAL STATUS: FAILED (Hard Hallucination)")

            print("--------------------------------------------------")

    if save_csv_path:
        out_df = pd.DataFrame(all_results)
        out_df.to_csv(save_csv_path, index=False)
        print(f"\nResults saved to {save_csv_path}")